In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')
print(df.shape)

(180519, 53)


In [2]:
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print("Duplicate rows:", df.duplicated().sum())

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64
Duplicate rows: 0


In [3]:
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'])
df['ship_date'] = pd.to_datetime(df['shipping date (DateOrders)'])

rename_map = {
    'Order Id': 'order_id', 'Days for shipping (real)': 'days_real',
    'Days for shipment (scheduled)': 'days_scheduled', 'Delivery Status': 'delivery_status',
    'Late_delivery_risk': 'late_flag', 'Shipping Mode': 'ship_mode',
    'Order Region': 'region', 'Market': 'market', 'Category Name': 'category',
    'Product Name': 'product', 'Product Card Id': 'product_id',
    'Order Item Quantity': 'qty', 'Sales': 'sales',
    'Customer Segment': 'segment', 'Order Status': 'order_status'
}
df = df.rename(columns=rename_map)
df = df[list(rename_map.values()) + ['order_date', 'ship_date']]

In [4]:
print(df['delivery_status'].value_counts(), '\n')
print(df['order_status'].value_counts())

delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64 

order_status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64


In [5]:
df = df[~df['order_status'].isin(['CANCELED', 'SUSPECTED_FRAUD'])].copy()
df['delay_days'] = df['days_real'] - df['days_scheduled']
df['is_late'] = (df['delay_days'] > 0).astype(int)
df['order_month'] = df['order_date'].dt.to_period('M').dt.to_timestamp()
print(df.shape)

(172765, 20)


In [6]:
df.to_csv('../data/clean_orders.csv', index=False)

In [7]:
print(f"Late delivery rate: {df['is_late'].mean()*100:.1f}%")
print(f"Avg delay when late: {df.loc[df['is_late']==1, 'delay_days'].mean():.1f} days")
print(f"Avg shipping time: {df['days_real'].mean():.1f} days")

Late delivery rate: 57.3%
Avg delay when late: 1.6 days
Avg shipping time: 3.5 days


In [8]:
print((df.groupby('ship_mode')['is_late'].mean().sort_values(ascending=False) * 100).round(1), '\n')
print((df.groupby('region')['is_late'].mean().sort_values(ascending=False) * 100).round(1))

ship_mode
First Class       100.0
Second Class       79.8
Same Day           47.9
Standard Class     39.8
Name: is_late, dtype: float64 

region
Central Africa     60.1
Western Europe     58.5
South of  USA      58.5
South Asia         58.4
East Africa        58.2
East of USA        58.2
Southeast Asia     58.0
West Asia          57.8
Eastern Europe     57.6
US Center          57.5
South America      57.2
Central America    57.1
North Africa       57.1
Southern Europe    56.8
West of USA        56.8
Eastern Asia       56.7
Central Asia       56.5
Oceania            56.3
Northern Europe    56.2
Southern Africa    56.1
Caribbean          55.5
West Africa        54.7
Canada             51.6
Name: is_late, dtype: float64


In [9]:
print(df.groupby('ship_mode')[['days_scheduled', 'days_real']].agg(['min', 'mean', 'max']).round(1))

               days_scheduled          days_real         
                          min mean max       min mean max
ship_mode                                                
First Class                 1  1.0   1         2  2.0   2
Same Day                    0  0.0   0         0  0.5   1
Second Class                2  2.0   2         2  4.0   6
Standard Class              4  4.0   4         2  4.0   6


In [10]:
rows = []
for mode, g in df.groupby('ship_mode'):
    for promise in range(int(g['days_scheduled'].iloc[0]), 7):
        rows.append({'ship_mode': mode, 'promise_days': promise,
                     'late_%': round((g['days_real'] > promise).mean() * 100, 1)})
print(pd.DataFrame(rows).pivot(index='ship_mode', columns='promise_days', values='late_%'))

promise_days       0      1     2     3     4     5    6
ship_mode                                               
First Class      NaN  100.0   0.0   0.0   0.0   0.0  0.0
Same Day        47.9    0.0   0.0   0.0   0.0   0.0  0.0
Second Class     NaN    NaN  79.8  59.8  39.8  19.8  0.0
Standard Class   NaN    NaN   NaN   NaN  39.8  20.2  0.0
